# 📓 Notebook 17 — Performance Optimization---**Phase 6 · Production Engineering**> Production systems need speed, efficiency, and cost control. Optimize every layer.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Semantic caching | Avoid redundant LLM calls || Token reduction | Spend fewer tokens per request || Streaming | Show results as they generate || Batch processing | Embed thousands efficiently |### 🏗️ Build: Optimized RAG Pipeline with Caching

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode, DEFAULT_EMBEDDING

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Semantic Caching### The IdeaIf a user asks "What is Python?" and another asks "Tell me about Python", the answer is the same. Cache it!| Cache Type | How | Speed | Accuracy ||-----------|-----|-------|----------|| **Exact match** | Hash the query string | O(1) | 100% || **Semantic** | Compare query embeddings | O(n) | ~95%+ |> **Interview Tip:** *"How would you reduce LLM costs in production?"*  > Semantic caching — if a similar query was answered recently, return the cached answer instead of calling the LLM again.

### 🖼️ How Semantic Caching Works

![Semantic Caching: Save Costs on Similar Queries](images/semantic_caching_flow.png)

#### 💡 Intuition: A Smart Memory for Your LLM

Normal caching only works for **exact matches** — "What is Python?" ≠ "Tell me about Python".

Semantic caching is smarter:
1. When a user asks a question, embed it into a vector
2. Compare that vector to all previously cached question vectors
3. If any cached question is very similar (cosine > 0.92), return the cached answer
4. Otherwise, call the LLM and save the result for next time

**The cost savings are real:**

| Metric | Without Cache | With Semantic Cache |
|--------|--------------|--------------------|
| LLM calls per day | 10,000 | ~4,000 (60% hit rate) |
| Monthly cost (GPT-4o-mini) | ~$45 | ~$18 |
| Average latency | 800ms | 120ms (cache hits) |

> **Trade-off:** Each cache lookup costs one embedding call. But embedding is ~100x cheaper than a full LLM call.

In [ ]:
class SemanticCache:    """Cache LLM responses using semantic similarity."""        def __init__(self, similarity_threshold=0.92, max_size=1000):        self.threshold = similarity_threshold        self.max_size = max_size        self.cache = []  # {"query": str, "embedding": list, "response": str, "timestamp": float}        self.hits = 0        self.misses = 0        def get(self, query):        """Check if a similar query has been cached."""        if not self.cache:            self.misses += 1            return None                query_emb = get_embedding(query)                best_match = None        best_score = 0                for entry in self.cache:            score = cosine_similarity(query_emb, entry["embedding"])            if score > best_score:                best_score = score                best_match = entry                if best_score >= self.threshold:            self.hits += 1            return {"response": best_match["response"], "score": best_score, "cached_query": best_match["query"]}                self.misses += 1        return None        def put(self, query, response):        """Store a query-response pair."""        embedding = get_embedding(query)        self.cache.append({            "query": query,            "embedding": embedding,            "response": response,            "timestamp": time.time(),        })        # Evict oldest if over max size        if len(self.cache) > self.max_size:            self.cache = self.cache[-self.max_size:]        def stats(self):        total = self.hits + self.misses        rate = self.hits / total * 100 if total > 0 else 0        print(f"  📊 Cache: {len(self.cache)} entries, {self.hits} hits, {self.misses} misses ({rate:.1f}% hit rate)")# Democache = SemanticCache(similarity_threshold=0.90)def cached_llm_call(query, cache):    """LLM call with semantic caching."""    # Check cache    cached = cache.get(query)    if cached:        print(f"  ⚡ CACHE HIT (score: {cached['score']:.3f}, original: \"{cached['cached_query'][:40]}...\")")        return cached["response"]        # Cache miss — call LLM    print(f"  🔄 CACHE MISS — calling LLM...")    response = litellm.completion(        model=DEFAULT_MODEL,        messages=[{"role": "user", "content": query}],        temperature=0, max_tokens=200,    )    answer = response.choices[0].message.content    cache.put(query, answer)    return answer# Test cachingqueries = [    "What is Python?",               # First call — miss    "Tell me about Python",           # Similar — should be a hit    "Python programming language",    # Similar — should be a hit    "What is JavaScript?",            # Different — miss    "Tell me about JavaScript",       # Similar to JS query — hit    "Explain Python to me",           # Similar to Python query — hit]print("⚡ Semantic Caching Demo")print("=" * 60)for q in queries:    print(f"\n❓ {q}")    answer = cached_llm_call(q, cache)    print(f"📝 {answer[:80]}...")cache.stats()

---## 3. Token Reduction Strategies

In [ ]:
import tiktokendef count_tokens(text, model="gpt-4o"):    try:        encoder = tiktoken.encoding_for_model(model)    except:        encoder = tiktoken.get_encoding("cl100k_base")    return len(encoder.encode(text))# Strategy 1: Prompt compression — remove unnecessary wordsdef compress_prompt(prompt, model=None):    """Use LLM to compress a prompt while preserving meaning."""    response = litellm.completion(        model=model or DEFAULT_MODEL,        messages=[{            "role": "user",            "content": f"Compress this text to be as short as possible while keeping ALL key information. Remove filler words and redundancy:\n\n{prompt}"        }],        temperature=0, max_tokens=len(prompt) // 2,    )    return response.choices[0].message.content# Demoverbose_prompt = """As a highly knowledgeable and experienced expert in the field of artificial intelligence and machine learning, I would like you to please provide me with a comprehensive and detailed explanation of how transformer neural network architectures work, including the self-attention mechanism, positional encoding, and the differences between encoder and decoder components. Please be thorough in your response."""compressed = compress_prompt(verbose_prompt)orig_tokens = count_tokens(verbose_prompt)comp_tokens = count_tokens(compressed)print(f"📦 Prompt Compression")print(f"   Original:   {orig_tokens} tokens")print(f"   Compressed: {comp_tokens} tokens")print(f"   Saved:      {orig_tokens - comp_tokens} tokens ({(1 - comp_tokens/orig_tokens)*100:.0f}%)")print(f"\n   Compressed: {compressed[:150]}...")

In [ ]:
# Strategy 2: Context pruning — remove low-relevance chunksdef prune_context(chunks, query, min_score=0.3):    """Remove chunks below relevance threshold."""    query_emb = get_embedding(query)    kept = []    removed = 0    for chunk in chunks:        chunk_emb = get_embedding(chunk)        score = cosine_similarity(query_emb, chunk_emb)        if score >= min_score:            kept.append({"text": chunk, "score": score})        else:            removed += 1        kept.sort(key=lambda x: x["score"], reverse=True)    print(f"  📋 Kept {len(kept)}/{len(chunks)} chunks (removed {removed} below {min_score})")    return [k["text"] for k in kept]# Demochunks = [    "Python is a high-level programming language.",           # Relevant    "The weather today is sunny and warm.",                   # Irrelevant    "Machine learning uses Python extensively.",              # Relevant      "The stock market closed at 35,000 points.",              # Irrelevant    "Python's scikit-learn library provides ML algorithms.",  # Very relevant]pruned = prune_context(chunks, "Python for machine learning", min_score=0.3)for c in pruned:    print(f"  ✅ {c}")

---## 4. Streaming Responses

In [ ]:
# Streaming — show tokens as they generatedef stream_response(prompt, model=None):    """Stream LLM response token by token."""    model = model or DEFAULT_MODEL        print("📡 Streaming response: ", end="", flush=True)        full_response = ""    response = litellm.completion(        model=model,        messages=[{"role": "user", "content": prompt}],        temperature=0.5,        max_tokens=200,        stream=True,    )        for chunk in response:        if chunk.choices[0].delta.content:            token = chunk.choices[0].delta.content            print(token, end="", flush=True)            full_response += token        print()  # New line    return full_response# Demoprint("⚡ Streaming Demo")print("=" * 50)result = stream_response("Explain caching in 3 sentences.")

---## 5. Batch Embedding

In [ ]:
# Batch embedding is significantly fastertexts = [f"Document number {i} about topic {chr(65 + i % 26)}" for i in range(50)]# Batchstart = time.time()batch_embs = [d["embedding"] for d in litellm.embedding(model=DEFAULT_EMBEDDING, input=texts).data]batch_time = time.time() - start# Sequential (first 10 only for comparison)start = time.time()seq_embs = [get_embedding(t) for t in texts[:10]]seq_time = time.time() - startprint(f"⚡ Embedding Performance")print(f"   Batch (50 texts): {batch_time:.2f}s ({batch_time/50*1000:.0f}ms per text)")print(f"   Sequential (10):  {seq_time:.2f}s ({seq_time/10*1000:.0f}ms per text)")est_seq_50 = seq_time / 10 * 50print(f"   Estimated sequential (50): {est_seq_50:.2f}s")print(f"   Speedup: {est_seq_50/batch_time:.1f}x")

---## 📝 Interview Quiz — Performance Optimization### Q1: What is semantic caching? How is it different from exact caching?<details><summary>💡 Answer</summary>**Exact caching:** Hash the query string → O(1) lookup. Only works for identical queries.**Semantic caching:** Embed the query → find similar cached queries via cosine similarity.- "What is Python?" and "Tell me about Python" → same cached response- Threshold: typically 0.90-0.95 cosine similarity- Trade-off: Embedding cost per lookup vs LLM call savings**When to use:** High-traffic applications with repeated/similar queries. A 50% cache hit rate can halve your LLM costs.</details>### Q2: List 5 ways to reduce LLM token consumption.<details><summary>💡 Answer</summary>1. **Prompt compression** — Remove filler words, use concise instructions2. **Context pruning** — Only include relevant retrieved chunks, drop low-score ones3. **Shorter system prompts** — Minimize instructions while preserving behavior4. **Smaller model for simple tasks** — Route easy questions to cheaper models5. **Response length limits** — Set `max_tokens` appropriately per use case6. **Caching** — Don't call LLM for repeated/similar queries7. **Batch processing** — Amortize system prompt tokens across multiple items</details>### Q3: When would you use streaming vs non-streaming?<details><summary>💡 Answer</summary>**Streaming (stream=True):**- User-facing chat interfaces (faster perceived latency)- Long responses where first tokens appear quickly- Real-time dashboards**Non-streaming (stream=False):**- Backend processing (pipeline steps)- When you need the full response before proceeding (tool calling)- JSON mode (need complete valid JSON)- Evaluation/testing</details>### Q4: How would you design a cost-optimized multi-model routing system?<details><summary>💡 Answer</summary>```User Query → Classifier (tiny model or rules)                ↓    ┌───────────┼──────────┐    ↓           ↓          ↓  Simple     Medium     Complex  (GPT-4o    (GPT-4o)   (GPT-4o   mini)                  with RAG)   $0.15/M    $2.50/M    $10/M```- **Classify** query complexity (rules or small model)- **Route** to cheapest model that can handle it- **Monitor** quality per tier and adjust routing thresholds- **Cache** across all tiersExpected savings: 60-80% cost reduction vs always using the best model.</details>

### 🖼️ Smart Model Routing

![Smart Model Routing for Cost Optimization](images/model_routing_diagram.png)

#### 💡 Why Use One Model When You Can Use Three?

Not every question needs GPT-4o. Most questions are simple and can be handled by a cheaper model.

**The routing strategy:**
1. A lightweight classifier (rules or tiny model) grades each query's complexity
2. Simple queries ("What time is it?") → cheapest model (GPT-4o-mini)
3. Medium queries ("Compare X vs Y") → standard model (GPT-4o)
4. Complex queries ("Analyze this report") → best model + RAG

This alone can **cut LLM costs by 60-80%** because most real-world queries are simple.

---## ✅ Notebook 17 Summary| Concept | Key Takeaway ||---------|-------------|| Semantic caching | Reuse answers for similar queries; 50%+ cost reduction || Token reduction | Compress prompts, prune context, use shorter system prompts || Streaming | Show tokens as they generate; better UX for chat || Batch embedding | 10-50x faster than sequential embedding || Model routing | Route simple queries to cheap models, complex to expensive |### 🏁 Course Complete! Move on to the **[Capstone Projects](./capstone/)** to apply everything you've learned.